# EVASPA Evaporative Fraction generic edge and model

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import xarray as xr

In [ ]:
from evaspa.edge import FlatEdge, LinearEdge
from evaspa.ef import EFModel

### Generate test data 

In [ ]:
def setup_data(
    albedo: tuple[float, float],
    dry: tuple[float, float],
    wet: tuple[float, float],
    size: int = 125,
    seed: int = 0,
) -> xr.Dataset:
    """
    Generate data for tests
    """

    def dry_edge(v: npt.ArrayLike) -> npt.NDArray:
        return dry[1] * v + dry[0]

    def wet_edge(v: npt.ArrayLike) -> npt.NDArray:
        return wet[1] * v + wet[0]

    np.random.seed(seed)
    albedo_arr = np.random.uniform(
        low=albedo[0], high=albedo[1], size=(size, size)
    )
    lon = np.linspace(start=1.0, stop=2.0, num=size, dtype=np.float32)
    lat = np.linspace(start=43.0, stop=44.0, num=size, dtype=np.float32)
    lst_func = np.vectorize(
        lambda x: np.random.uniform(wet_edge(x), dry_edge(x))
    )
    lst_arr = lst_func(albedo_arr)
    valid_arr = np.random.choice([0, 1], size=(size, size), p=[0.2, 0.8])
    ef_arr = (dry_edge(albedo_arr) - lst_arr) / (
        dry_edge(albedo_arr) - wet_edge(albedo_arr)
    )
    return xr.Dataset(
        data_vars={
            "albedo": (["lon", "lat"], albedo_arr),
            "lst": (["lon", "lat"], lst_arr),
            "valid": (["lon", "lat"], valid_arr),
            "ef": (["lon", "lat"], ef_arr),
        },
        coords={
            "lon": ("lon", lon),
            "lat": ("lat", lat),
        },
        attrs={"description": "Test data"},
    )

In [ ]:
# Generate data
data = setup_data(albedo=(0.0, 0.6), dry=(330.0, -13.0), wet=(300.0, 30.0))

In [ ]:
data

In [ ]:
def plot(data: xr.Dataset, model: EFModel = None, save: bool = False):
    """
    Plot
    """
    # dimensions
    # gridspec inside gridspec
    fig = plt.figure(constrained_layout=True, figsize=(4, 3))

    ax = plt.gca()

    var = data["albedo"].data
    lst = data["lst"].data

    # plot all points
    ax.scatter(var, lst, s=20, alpha=1, color="moccasin", clip_on=False)

    # plot edges
    if model is not None:
        # edges coordinates
        var_max = np.round(np.ceil(np.nanmax(var) * 10) / 10, decimals=1)
        var_min = np.round(np.floor(np.nanmin(var) * 10) / 10, decimals=1)

        var_arr = np.linspace(var_min, var_max, num=20)
        tw_arr = model.twet(var_arr)
        td_arr = model.tdry(var_arr)

        ax.plot(
            var_arr,
            td_arr,
            color="tab:red",
            linewidth=3,
            label="dry edge",
            zorder=1,
        )
        ax.plot(
            var_arr,
            tw_arr,
            color="tab:blue",
            linewidth=3,
            label="wet edge",
            zorder=1,
        )

    # Labels
    ax.set_xlabel("Variable")
    ax.set_ylabel("Land Surface Temperature K")

    # title
    ax.set_title("Evaporative fraction method", fontsize=12)

    # savefig
    if save:
        destfile = "ef.png"
        fig.savefig(destfile)

    plt.show()

In [ ]:
plot(data)

## Edge definition

In [ ]:
config1 = '{"interval_type":"size","interval_nb":20,"percentile":[98,100],"selection":"max"}'

In [ ]:
edge1 = LinearEdge.model_validate_json(config1)

In [ ]:
edge1.fit(data["albedo"], data["lst"])

In [ ]:
edge1

In [ ]:
edge2 = FlatEdge.model_validate_json('{"selection":"max"}')

In [ ]:
edge2.fit(data["albedo"], data["lst"])

In [ ]:
edge2

In [ ]:
edge3 = FlatEdge.model_validate_json('{"selection":"min"}')
edge3.fit(data["albedo"], data["lst"])
edge3

## EF model definition

In [ ]:
config_model1 = {
    "name": "model1",
    "dry_edge": {
        "type": "LinearEdge",
        "config": {
            "interval_type": "size",
            "interval_nb": 20,
            "percentile": [98, 100],
            "selection": "median",
        },
    },
    "wet_edge": {
        "type": "LinearEdge",
        "config": {
            "interval_type": "size",
            "interval_nb": 20,
            "percentile": [0, 2],
            "selection": "median",
        },
    },
    "var": "albedo",
}

In [ ]:
model1 = EFModel.create(config_model1)

In [ ]:
model1.fit(data)

In [ ]:
model1.fit(data, mask="valid")

In [ ]:
model1.compute(data)

In [ ]:
model1.compute(data, mask="valid")

In [ ]:
plot(data, model=model1)

In [ ]:
print(model1)

In [ ]:
model1

In [ ]:
model1.to_dict()